# مصنّف الحيوانات — الأقسام 1 إلى 4: تجهيز البيانات (الإصدار 4)

**قرار هذا الإصدار: استبعاد الداتا الثانية d2 بالكامل.** ظهر من صورك أنها **نسخة مولَّدة بالتدوير والإمالة** (حواف سوداء وصور مائلة ومقلوبة). هذا يعني:
- **نسخ متعددة من الصورة الأصلية نفسها**، فتقع نسخة في train وأخرى في test، وتصبح دقة الاختبار مضخّمة (كمن رأى أسئلة الامتحان قبله). فحص التشابه عندنا لا يلتقط الصور المُدارة أو المائلة.
- **الحواف السوداء والإمالة بصمة** قد يتعلمها المودل بدل الحيوان.
- محاولة تصحيح الاتجاه بنموذج ImageNet لم تكن موثوقة (صور كثيرة بقيت جانبية).

**ما تغيّر:**
1. d2 غير محمّلة ولا مستخدمة، ولذلك سقطت `rabbit` و`chicken` لأنهما منها فقط. بدائلهما من d3: **`hare` (أرنب بري)** و**`penguin`**. غيّرهما من قائمة `TARGET_CLASSES` إن شئت؛ الكود يتوقف برسالة واضحة إذا لم يجد اسمًا.
2. حُذفت خلية تصحيح الاتجاه (3.4) لأنها لم تعد لازمة.
3. **اختيار الصور صار ثابتًا لكل فئة** (بذرة لكل فئة ومصدر)، فتبقى الأسماء مستقرة بين إعادات البناء.
4. القائمة السوداء تضم الآن صور الفلامنغو الصناعية (مجسم ثلاثي الأبعاد وفلامنغو بلاستيكي) إضافة إلى الفيل المرسوم وظل الفراشة.
5. حفظ النسخة باسم `data_v4.zip`.

**قبل التشغيل:** T4 GPU. **الناتج:** `split/` (train/val/test لـ24 فئة) و`manifest.csv` ونسخة في Drive.

## القسم 1 — الإعدادات

In [ ]:
!pip -q install kagglehub

import os, time, random, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.fft import dct
from sklearn.model_selection import train_test_split

Image.MAX_IMAGE_PIXELS = 60_000_000   # حماية من الصور العملاقة

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# الفئات الـ24 — ثبّتها ولا تغيّرها بعد اليوم
TARGET_CLASSES = [
    # فئات غنية (تأتي من أكثر من مصدر)
    "cat", "dog", "horse", "cow", "sheep", "goat", "deer", "hare", "penguin", "elephant", "lion",
    # فئات من داتا الـ90
    "antelope", "bear", "bison", "butterfly", "chimpanzee", "crow", "dolphin",
    "donkey", "duck", "eagle", "flamingo", "fox", "zebra",
]
CLASS_MAP    = {}                                  # توحيد أسماء المجلدات بين المصادر إن لزم
EXCLUDE_DIRS = {"inf", "nightvision", "__macosx"}  # مجلدات ليست فئات حيوانات

DATASETS = {   # d2 (mohamedramadan2040/animals) مستبعدة: نسخة مولَّدة بالتدوير والإمالة
    "d1": "antobenedetti/animals",
    "d3": "iamsouravbanerjee/animal-image-dataset-90-different-animals",
}

CAP       = 200    # أقصى عدد صور لكل فئة
MAX_PER_SRC     = {"d1": 100, "d3": 150}  # أقصى عدد لكل فئة من المصدر الواحد
MIN_SIDE_BY_SRC = {"d1": 100, "d3": 64}   # أصغر بُعد مقبول (d1 فيها صور منخفضة الدقة تظهر مبكسلة)
MAX_SIDE  = 384    # نصغّر الصور المحفوظة؛ التدريب سيكون على 224
DUP_BY_SRC = {"d1": 4, "d3": 4}  # مسافة Hamming (من 64) لاعتبار صورتين مكررتين
LEAK_DIST = 8      # أي صورة في val/test أقرب من هذا لصورة في train تُنقل إلى train
GENERIC_DIRS = {"images", "image", "imgs", "img", "train", "val", "valid", "validation", "test", "data", "dataset"}
VAL_FRAC, TEST_FRAC = 0.15, 0.15

PROJECT   = Path("/content/project")
DATA      = PROJECT / "data"
MERGED    = DATA / "merged"
SPLIT_DIR = DATA / "split"
REPORTS   = PROJECT / "reports"
DRIVE_DIR = Path("/content/drive/MyDrive/animal-classifier")
ARCHIVE   = DRIVE_DIR / "data_v4.zip"
BLACKLIST = DRIVE_DIR / "blacklist.txt"   # صور محذوفة نهائيًا (تبقى بين إعادات البناء)
REBUILD   = False  # True = تجاهل نسخة Drive وابنِ من الصفر

assert len(TARGET_CLASSES) == len(set(TARGET_CLASSES)) == 24

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

BUILD = REBUILD or not ARCHIVE.exists()
if BUILD:
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    for d in (MERGED, REPORTS):
        d.mkdir(parents=True, exist_ok=True)
    print("🔨 سنبني البيانات من الصفر — شغّل الأقسام 2 إلى 4")
else:
    shutil.unpack_archive(str(ARCHIVE), str(PROJECT))
    print("✅ استرجعت البيانات من Drive — انتقل مباشرة إلى القسم 5")

def skip_if_restored():
    if not BUILD:
        raise SystemExit("البيانات مسترجعة من Drive — تجاوز هذه الخلية")

## القسم 2 — تحميل الداتا من Kaggle
حوالي 2.5 GB، تأخذ بضع دقائق.

In [ ]:
skip_if_restored()
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as e:
    print("لم أجد Secrets، سجّل الدخول يدويًا:", e)
    import kagglehub
    kagglehub.login()

import kagglehub
ROOTS = {}
for src, slug in DATASETS.items():
    t = time.time()
    ROOTS[src] = Path(kagglehub.dataset_download(slug))
    print(f"✓ {src}: {slug}  ({time.time() - t:.0f}s)")

## القسم 3 — المسح والدمج والتنظيف

**3.1 جرد ما في الداتا.** كل مجلد يحتوي صورًا مباشرة يُعتبر فئة، ويُوحَّد اسمه (أحرف صغيرة، و`hen` يصبح `chicken`). تُستبعد مجلدات `inf` و`NightVision`.

In [ ]:
skip_if_restored()
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def norm_name(name):
    n = name.strip().lower().replace(" ", "_")
    return CLASS_MAP.get(n, n)

def class_of(dirpath, root):
    # اسم الفئة = أقرب مجلد أب ليس اسمًا عامًا (images/train/val...)
    p, root = Path(dirpath), Path(root)
    while p != root and p.name.lower() in GENERIC_DIRS:
        p = p.parent
    return None if p == root else norm_name(p.name)

def scan(root, source):
    rows = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() not in EXCLUDE_DIRS]
        imgs = [f for f in filenames
                if Path(f).suffix.lower() in IMG_EXT and not f.startswith(".")]
        cls = class_of(dirpath, root) if imgs else None
        if cls:
            rows += [(source, cls, os.path.join(dirpath, f)) for f in imgs]
    return rows

rows = []
for src, root in ROOTS.items():
    rows += scan(root, src)
inv = pd.DataFrame(rows, columns=["source", "cls", "src_path"])
pivot = inv.pivot_table(index="cls", columns="source", values="src_path",
                        aggfunc="count", fill_value=0)

found   = [c for c in TARGET_CLASSES if c in pivot.index]
missing = [c for c in TARGET_CLASSES if c not in pivot.index]
print(f"الصور الخام: {len(inv):,} | كل الفئات المكتشفة: {len(pivot)} | المستهدفة الموجودة: {len(found)}/24")
display(pivot.loc[found])
if missing:
    print("الأسماء المتاحة:", sorted(pivot.index))
    raise AssertionError(f"فئات مستهدفة غير موجودة: {missing}")

**3.1ب القائمة السوداء.** صور حذفتها يدويًا من قبل (رسم قلم لفيل، ظل فراشة). تُحفظ في Drive بمفتاح ثابت (المصدر + مسار الصورة داخله)، فلا تعود عند أي إعادة بناء.

In [ ]:
skip_if_restored()
import re, zipfile

def src_key(src, path):
    return src + ":" + re.split(r"/versions/\d+/", str(path), maxsplit=1)[-1]

BLACKLIST_KEYS = set(BLACKLIST.read_text().splitlines()) if BLACKLIST.exists() else set()

# صور حدّدتَها في بناءات سابقة (اسم الملف داخل manifest ذلك البناء) — تُضاف مرة واحدة
OLD_BAD = {
    "data_v2.zip": ["elephant_0099_d1.jpg", "butterfly_0002_d3.jpg"],               # رسم قلم، ظل فراشة
    "data_v3.zip": ["flamingo_0030_d3.jpg", "flamingo_0004_d3.jpg", "flamingo_0029_d3.jpg"],  # مجسم 3D، فلامنغو بلاستيك
}
for zname, bad in OLD_BAD.items():
    zp = DRIVE_DIR / zname
    if not zp.exists():
        print("غير موجود (تخطّيت):", zname); continue
    with zipfile.ZipFile(zp) as z:
        old = pd.read_csv(z.open("data/manifest.csv"))
    hit = old[old.file.isin(bad)]
    BLACKLIST_KEYS.update(src_key(s_, p) for s_, p in zip(hit.source, hit.src_path))
    print(f"{zname}: أُضيف {len(hit)} من {len(bad)}")
BLACKLIST.write_text("\n".join(sorted(BLACKLIST_KEYS)))
print("حجم القائمة السوداء:", len(BLACKLIST_KEYS))

**3.2 التنظيف وحذف المكرر وتطبيق السقف.** لكل صورة:
1. تُفتح وتُصحَّح من EXIF وتُحوَّل إلى RGB (الشفافية تصبح خلفية بيضاء)، وتُرفض التالفة والصغيرة.
2. تُحسب **بصمتها البصرية** (perceptual hash). إذا كانت شبه مطابقة لصورة محفوظة:
   - **من نفس الفئة:** تُحذف كتكرار.
   - **من فئة أخرى:** تُحذف الاثنتان، لأن التسمية متعارضة.
3. تُؤخذ الصور من المصادر **بالتناوب**، فتأتي صور القطط مثلًا من الداتا الثلاث، لا من واحدة فقط.

In [ ]:
skip_if_restored()
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)  # عدّ البتات

def load_rgb(path):
    with Image.open(path) as im:
        im.draft("RGB", (MAX_SIDE * 2, MAX_SIDE * 2))   # يسرّع فك ملفات JPEG الكبيرة
        try:
            im = ImageOps.exif_transpose(im)             # صور الجوال المقلوبة
        except Exception:
            pass
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            bg = Image.new("RGB", im.size, (255, 255, 255))
            bg.paste(im, mask=im.getchannel("A"))
            return bg
        return im.convert("RGB")

def phash64(im):
    g = np.asarray(im.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float64)
    d = dct(dct(g, axis=0), axis=1)[:8, :8]
    bits = (d > np.median(d)).flatten()
    return np.packbits(bits).view(">u8")[0].astype(np.uint64)

kept_hash = np.zeros(len(TARGET_CLASSES) * CAP + 16, dtype=np.uint64)
records, conflicts, stats = [], set(), Counter()

def nearest(h):
    n = len(records)
    if n == 0:
        return -1, 64
    x = np.bitwise_xor(kept_hash[:n], h).view(np.uint8).reshape(n, 8)
    dist = POP[x].sum(axis=1)
    i = int(dist.argmin())
    return i, int(dist[i])

def round_robin(lists):
    iters = [iter(l) for l in lists if l]
    while iters:
        for it in list(iters):
            try:
                yield next(it)
            except StopIteration:
                iters.remove(it)

t0 = time.time()
for cls in TARGET_CLASSES:
    sub = inv[inv.cls == cls]
    per_src = []
    for src in DATASETS:
        lst = sub[sub.source == src].src_path.tolist()
        random.Random(f"{SEED}|{cls}|{src}").shuffle(lst)   # ترتيب ثابت لكل فئة ومصدر
        per_src.append([(src, p) for p in lst])
    (MERGED / cls).mkdir(parents=True, exist_ok=True)

    kept, src_kept = 0, Counter()
    for src, path in round_robin(per_src):
        if kept >= CAP:
            break
        if src_kept[src] >= MAX_PER_SRC[src]:
            continue
        if src_key(src, path) in BLACKLIST_KEYS:
            stats["blacklisted"] += 1
            continue
        try:
            im = load_rgb(path)
        except Exception:
            stats["unreadable"] += 1
            continue
        if min(im.size) < MIN_SIDE_BY_SRC[src]:
            stats["too_small"] += 1
            continue
        h = phash64(im)
        i, d = nearest(h)
        if d <= DUP_BY_SRC[src]:
            if records[i]["cls"] == cls:
                stats["duplicate"] += 1
            else:
                stats["cross_class_conflict"] += 1
                conflicts.add(i)
            continue
        orig_min = min(im.size)
        im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
        sat = float(np.asarray(im.convert("HSV"))[..., 1].mean() / 255)
        out = MERGED / cls / f"{cls}_{kept:04d}_{src}.jpg"
        im.save(out, "JPEG", quality=92)
        kept_hash[len(records)] = h
        records.append({"file": out.name, "path": str(out), "cls": cls, "source": src,
                        "src_path": path, "phash": f"{int(h):016x}", "saturation": round(sat, 3),
                        "min_side": orig_min})
        kept += 1
        src_kept[src] += 1
    print(f"{cls:<11} {kept:>4}")

for i in conflicts:                                  # احذف الطرف المحفوظ من كل تعارض
    Path(records[i]["path"]).unlink(missing_ok=True)
manifest = pd.DataFrame([r for j, r in enumerate(records) if j not in conflicts])
manifest.to_csv(DATA / "manifest.csv", index=False)

print(f"\n⏱ {time.time() - t0:.0f}s | محفوظة: {len(manifest)} | {dict(stats)}")

**3.3 ملخص التوزيع.** يُحفظ الرسم في `reports/` لتستخدمه في صفحة 2 من الموقع.

In [ ]:
skip_if_restored()
summary = (manifest.pivot_table(index="cls", columns="source", values="file",
                                aggfunc="count", fill_value=0)
           .reindex(TARGET_CLASSES, fill_value=0))
summary["total"] = summary.sum(axis=1)
display(summary)
print("إجمالي الصور:", len(manifest))
low = summary[summary.total < 50].index.tolist()
if low:
    print("⚠️ فئات أقل من 50 صورة:", low)

print("\nأصغر بُعد (بكسل) للصورة الأصلية، حسب المصدر:")
display(manifest.groupby("source").min_side.describe(percentiles=[.05, .25, .5]).round(0))

ax = summary.drop(columns="total").plot(kind="bar", stacked=True, figsize=(14, 5))
ax.set_title("Images per class by source")
ax.set_ylabel("images")
plt.tight_layout()
plt.savefig(REPORTS / "class_distribution.png", dpi=150)
plt.show()

## القسم 4 — المراجعة البصرية والتقسيم

**4.1 راجع كل فئة بعينك.** اكتب أسماء الصور الخاطئة (المكتوبة فوق كل صورة) في الخلية 4.3.

**احذف:** حيوانًا من فئة خاطئة، رسمًا أو لعبة أو مجسمًا (فلامنغو بلاستيك، رسم ثلاثي الأبعاد)، صورة فيها عدة حيوانات مختلفة، صورة لا يظهر فيها الحيوان.

**لا تحذف:** الصور الصعبة الصحيحة (بعيدة، جزء من الحيوان، زاوية غريبة). هذه تعلّم المودل التعميم.

In [ ]:
skip_if_restored()
def show_grid(df, title, n=12, seed=0):
    if len(df) == 0:
        return
    sample = df.sample(min(n, len(df)), random_state=seed)
    cols = 6
    nrows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(16, 2.9 * nrows))
    axes = np.array(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, sample.iterrows()):
        ax.imshow(Image.open(r.path))
        ax.set_title(r.file, fontsize=7)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

for cls in TARGET_CLASSES:
    sub = manifest[manifest.cls == cls]
    show_grid(sub, f"{cls}  ({len(sub)} images)")

**4.2 الصور الأقل ألوانًا.** تكشف صور الرؤية الليلية أو الأبيض والأسود التي تسربت إلى فئات عادية.

In [ ]:
skip_if_restored()
show_grid(manifest.nsmallest(18, "saturation"), "Lowest-saturation images", n=18)

**4.2ب مقارنة المصادر.** كل صف = مصدر مختلف لنفس الفئة. انظر: هل صور d1 (وجوه مقرّبة غالبًا) تختلف كثيرًا عن صور d3؟ هذا يحدد خطر أن يتعلم المودل "أسلوب المصدر" بدل الحيوان.

In [ ]:
skip_if_restored()
def compare_sources(cls, n=6):
    srcs = ["d1", "d3"]
    fig, axes = plt.subplots(2, n, figsize=(2.8 * n, 6))
    for r, s_ in enumerate(srcs):
        sub = manifest[(manifest.cls == cls) & (manifest.source == s_)]
        sub = sub.sample(min(n, len(sub)), random_state=1)
        for c in range(n):
            ax = axes[r, c]; ax.axis("off")
            if c < len(sub):
                ax.imshow(Image.open(sub.iloc[c].path))
            if c == 0:
                ax.set_title(f"{cls} - {s_}  ({int(((manifest.cls == cls) & (manifest.source == s_)).sum())} imgs)",
                             loc="left", fontsize=11)
    plt.tight_layout(); plt.show()

for c in ["cat", "dog", "horse"]:
    compare_sources(c)

**4.3 احذف الصور الخاطئة.** الصق الأسماء في القائمة ثم شغّل الخلية. تُسجَّل الصور المحذوفة في القائمة السوداء على Drive فلا تعود بعد أي إعادة بناء.

In [ ]:
skip_if_restored()
BAD_FILES = [
    # "cat_0012_d3.jpg",
]

removed = manifest[manifest.file.isin(BAD_FILES)]
for p in removed.path:
    Path(p).unlink(missing_ok=True)
if len(removed):                                   # تذكّرها نهائيًا لأي إعادة بناء
    BLACKLIST_KEYS.update(src_key(s_, p) for s_, p in zip(removed.source, removed.src_path))
    BLACKLIST.write_text("\n".join(sorted(BLACKLIST_KEYS)))
manifest = manifest[~manifest.file.isin(BAD_FILES)].reset_index(drop=True)
manifest.to_csv(DATA / "manifest.csv", index=False)

not_found = set(BAD_FILES) - set(removed.file)
print(f"حُذفت {len(removed)} صورة | المتبقي {len(manifest)}")
if not_found:
    print("⚠️ أسماء غير موجودة (ربما حُذفت سابقًا):", not_found)

**4.4 التقسيم الثابت 70/15/15.** متوازن حسب الفئة وبـ`SEED` ثابت، ثم **فحص تسريب**: يتأكد أنه لا توجد صورة في val أو test شبه مطابقة لصورة في train.

In [ ]:
skip_if_restored()
train_df, tmp = train_test_split(manifest, test_size=VAL_FRAC + TEST_FRAC,
                                 stratify=manifest.cls, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
                                   stratify=tmp.cls, random_state=SEED)

def to_u64(hexes):
    return np.array([int(h, 16) for h in hexes], dtype=np.uint64)

def nearest_train_dist(df, train_hex):
    b = to_u64(train_hex)
    out = []
    for h in to_u64(df.phash):
        x = np.bitwise_xor(b, h).view(np.uint8).reshape(len(b), 8)
        out.append(int(POP[x].sum(axis=1).min()))
    return np.array(out)

# كنس التسريب: أي صورة في val/test قريبة من صورة في train تنتقل إلى train
moved = {}
for name, df in (("val", val_df), ("test", test_df)):
    d = nearest_train_dist(df, train_df.phash)
    leak = df[d <= LEAK_DIST]
    moved[name] = leak.source.value_counts().to_dict()
    train_df = pd.concat([train_df, leak])
    if name == "val":
        val_df = df[d > LEAK_DIST]
    else:
        test_df = df[d > LEAK_DIST]
print("نُقل من val/test إلى train لأنه شبه مكرر:", moved)

d_val, d_test = nearest_train_dist(val_df, train_df.phash).min(), nearest_train_dist(test_df, train_df.phash).min()
print(f"أقل مسافة val↔train: {d_val} | test↔train: {d_test} (يجب أن تكون > {LEAK_DIST})")
assert min(d_val, d_test) > LEAK_DIST, "تسريب! صورة شبه مكررة بين التقسيمات"

where = {}
for split, df in {"train": train_df, "val": val_df, "test": test_df}.items():
    for f, c, p in zip(df.file, df.cls, df.path):
        dst = SPLIT_DIR / split / c / f
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(p, dst)
        where[f] = (split, str(dst))
manifest["split"] = manifest.file.map(lambda f: where[f][0])
manifest["path"] = manifest.file.map(lambda f: where[f][1])
manifest.to_csv(DATA / "manifest.csv", index=False)
shutil.rmtree(MERGED, ignore_errors=True)

table = (pd.crosstab(manifest.cls, manifest.split)
         .reindex(TARGET_CLASSES, fill_value=0)[["train", "val", "test"]].astype(int))
display(table)
print("المجموع:", table.sum().to_dict())
print("\nالمصدر × التقسيم:")
display(pd.crosstab(manifest.source, manifest.split)[["train", "val", "test"]])

**4.5 احفظ نسخة في Google Drive.** في الجلسات القادمة تُسترجع تلقائيًا من الخلية 1.3 بدل إعادة التحميل.

In [ ]:
skip_if_restored()
t = time.time()
local_zip = shutil.make_archive("/content/data_v4", "zip", root_dir=PROJECT)
shutil.copy(local_zip, ARCHIVE)
print(f"✅ حُفظت في {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.0f} MB, {time.time() - t:.0f}s)")

## انتهت الأقسام 1–4

أرسل لي هذه المخرجات قبل القسم 5:
1. سطر الإحصاءات في نهاية **3.2** (وفيه `blacklisted`)، وجدول **3.3** (يجب أن تظهر d1 وd3 فقط).
2. صور **4.2ب** (مقارنة d1 وd3).
3. أسماء أي صور حذفتها في **4.3** وسببها.
4. سطر "نُقل من val/test" وجدول **المصدر × التقسيم** من **4.4**.